# V3 — robustness and validation error analysis

This notebook reads artifacts created by `python -m src.pipeline_v3`. E1, E4 and E6 have fixed V1/V2 parameters. Five stratified folds are constructed exclusively from official train. The validation set is analyzed separately using saved V2 predictions. There is no V2/V3 test evaluation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
root = Path.cwd()
if not (root / 'configs/experiments_v3.yaml').exists(): root = root.parent
out = root / 'results/v3'
summary = json.loads((out / 'metrics/cv_summary.json').read_text())
display(pd.DataFrame(summary['summaries']))
display(Image(filename=str(out / 'figures/cv_comparison.png')))

## Per-class validation metrics and errors

Each official validation class has only ten images, so class recall has 0.1 steps. Directed pairs count true class → predicted class. The displayed images are observations; possible effects of color, lighting, scale, shape or background are hypotheses unless supported by controlled experiments.

In [ ]:
for experiment in ['E4', 'E5', 'E6']:
    print(experiment, 'lowest F1:')
    display(pd.read_csv(out / f'metrics/{experiment}_per_class.csv').sort_values(['f1','class_id']).head(10))
    print('Most frequent directed confusion pairs:')
    display(pd.read_csv(out / f'metrics/{experiment}_confusion_pairs.csv').head(10))
display(Image(filename=str(out / 'figures/E6_per_class_f1.png')))
display(Image(filename=str(out / 'figures/E6_confusion_examples.png')))

## Observations from the saved validation examples

E6 has 22 validation classes with F1 = 0, based on ten images per class. The largest directed confusions are 24 → 11 and 94 → 32 (four images each). In the examples, IDs 6814, 6819 and 6834 show red/pink class-24 flowers among green leaves; class-94 IDs 7336 and 7386 differ in object scale and background. Class-18 IDs 4260 and 4296 differ in apparent illumination and crop. Color, shape, scale, lighting and background may affect the features, but no controlled experiment establishes these as causes of the errors.

## Interpretation and protocol

CV folds reuse overlapping training subsets, and fixed hyperparameters were chosen on the separate validation set. Fold standard deviation is descriptive, not a confidence interval. Repeated PCA seeds vary the randomized solver, not the split or the data. Preserve the existing test budget: document and freeze model, preprocessing and decision rule before a single explicitly authorized final E6 test evaluation; distinguish the previously inspected V1 test result from an untouched holdout.